# 07 CURE with a bigger sample

In [1]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("..")          # cf_common.py and cure.py are in the folder above

from cure import cure
from cf_common import load_data, evaluate_clustering, save_results, assign_to_nearest

data = load_data()
SAMPLE_SIZES = [20_000, 30_000]
K_VALUES = [10, 20, 40, 80]
N_REPRESENTATIVES = 8
SHRINK = 0.8
MIN_CLUSTER_SIZE = 10

476,451 users, 120,650 items, 10,000 users are evaluated


In [2]:
summaries, per_user = [], []
for sample_size in SAMPLE_SIZES:
    if sample_size > data.n_users:          # only happens in a quick test with few users
        print(f"skipped sample {sample_size:,}: only {data.n_users:,} users")
        continue
    sample = np.random.default_rng(579).choice(data.n_users, size=sample_size, replace=False)
    start = time.perf_counter()
    representatives = cure(data.embedding[sample], K_VALUES, N_REPRESENTATIVES, SHRINK, MIN_CLUSTER_SIZE)
    cure_seconds = time.perf_counter() - start
    print(f"--- sample {sample_size:,}: CURE {cure_seconds:.0f}s")
    for k in K_VALUES:
        start = time.perf_counter()
        rep_points, rep_owner = representatives[k]
        labels = rep_owner[assign_to_nearest(data.embedding, rep_points)]
        summary, users = evaluate_clustering(data, "cure", k, labels, cure_seconds + time.perf_counter() - start,
                                             extra={"sample size": sample_size})
        summaries.append(summary)
        per_user.append(users)

results = save_results("cure_sample_size", summaries, per_user, folder="results/analyses")

--- sample 20,000: CURE 189s


cure             k=10   P@10 = 0.1260   clustering  189.1s   CF   28.3s


cure             k=20   P@10 = 0.1239   clustering  189.2s   CF   24.3s


cure             k=40   P@10 = 0.1199   clustering  189.4s   CF   24.8s


cure             k=80   P@10 = 0.1148   clustering  190.3s   CF   29.7s


--- sample 30,000: CURE 635s


cure             k=10   P@10 = 0.1270   clustering  635.6s   CF   53.1s


cure             k=20   P@10 = 0.1237   clustering  635.7s   CF   39.8s


cure             k=40   P@10 = 0.1196   clustering  635.9s   CF   29.4s


cure             k=80   P@10 = 0.1161   clustering  636.1s   CF   28.6s


saved results\analyses\cure_sample_size.csv and results\analyses\cure_sample_size_per_user.csv


## Results (the 10,000 rows are from `02_cure`)

In [3]:
main = pd.read_csv("results/cure.csv").assign(**{"sample size": 10_000})
both = pd.concat([main, results], ignore_index=True)
both["total min"] = (both["clustering s"] + both["CF s (all users, estimated)"]) / 60
both[["sample size", "k", "clusters", "largest cluster %", "avg neighbour pool", "P@10", "clustering s", "total min"]].round(4)

,sample size,k,clusters,largest cluster %,avg neighbour pool,P@10,clustering s,total min
0,10000,10,9,34.9316,99910.9724,0.1257,33.1959,20.2092
1,10000,20,14,32.9929,81934.1852,0.1228,33.3108,18.5639
2,10000,40,24,31.0462,71129.9358,0.1202,33.5145,18.1384
3,10000,80,46,27.5588,51653.9578,0.1153,33.8570,16.2526
4,20000,10,9,40.4123,111364.8133,0.1260,189.0738,25.5908
5,20000,20,14,32.8216,82649.0574,0.1239,189.1544,22.4766
6,20000,40,24,24.7742,57449.5216,0.1199,189.3589,22.8388
7,20000,80,46,20.2415,38820.3773,0.1148,190.2961,26.7896
8,30000,10,8,53.9226,158794.5649,0.1270,635.6093,52.7867
9,30000,20,16,33.6723,86486.4982,0.1237,635.6539,42.1597
